# Extra-Inning Strategy in MLB

This notebook analyzes offensive and defensive strategy under MLB's automatic-runner extra-inning rule.

The workflow is organized into five parts:

1. **Build the extra-inning dataset**
2. **Identify bunt attempts and game outcomes**
3. **Add hitter quality**
4. **Estimate bunt and intentional-walk effects**
5. **Build an empirical strategy engine and bootstrap uncertainty**

The goal is to maximize **win probability**, not simply expected runs.

## 1. Setup

The notebook assumes daily Statcast CSV files already exist in `data/raw/`.

If they do not, set `DOWNLOAD_RAW_DATA = True` in the next cell. The download code is separated from the analysis so the notebook can be rerun without repeatedly querying Baseball Savant.

In [ ]:
from pathlib import Path
from datetime import datetime, timedelta
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import statsmodels.api as sm
import statsmodels.formula.api as smf

from scipy.stats import beta, fisher_exact
from statsmodels.stats.proportion import proportions_ztest

from pybaseball import statcast

# ---------------------------------------------------------------------
# Project configuration
# ---------------------------------------------------------------------
START_DATE = "2021-04-01"
END_DATE = "2025-10-01"

DATA_DIR = Path("data")
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
FIGURE_DIR = Path("figures")
RESULTS_DIR = Path("results")

for folder in [RAW_DIR, PROCESSED_DIR, FIGURE_DIR, RESULTS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", None)

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

### Optional: download daily Statcast files

In [ ]:
DOWNLOAD_RAW_DATA = False

if DOWNLOAD_RAW_DATA:
    start = datetime.strptime(START_DATE, "%Y-%m-%d")
    end = datetime.strptime(END_DATE, "%Y-%m-%d")
    current_date = start

    downloaded = 0
    skipped = 0
    failed = []

    while current_date < end:
        date_string = current_date.strftime("%Y-%m-%d")
        next_date = current_date + timedelta(days=1)
        next_date_string = next_date.strftime("%Y-%m-%d")

        output_file = RAW_DIR / f"statcast_{date_string}.csv"

        # Do not redownload dates that already exist locally.
        if output_file.exists():
            skipped += 1
            current_date = next_date
            continue

        try:
            daily_data = statcast(
                start_dt=date_string,
                end_dt=next_date_string
            )
            daily_data.to_csv(output_file, index=False)
            downloaded += 1

        except Exception as exc:
            failed.append((date_string, str(exc)))

        current_date = next_date
        time.sleep(1)

    print(f"Downloaded: {downloaded}")
    print(f"Skipped existing files: {skipped}")
    print(f"Failed dates: {len(failed)}")

## 2. Build the extra-inning pitch dataset

Only a small set of Statcast fields is needed for this project. Reading just these columns keeps memory usage manageable.

The resulting `extra` DataFrame contains every pitch from innings 10 and later.

In [ ]:
EXTRA_COLUMNS = [
    # Game / inning identifiers
    "game_date", "game_pk", "home_team", "away_team",
    "inning", "inning_topbot", "outs_when_up",
    "at_bat_number", "pitch_number",

    # Players and handedness
    "batter", "pitcher", "stand", "p_throws",

    # Base state
    "on_1b", "on_2b", "on_3b",

    # Plate-appearance / pitch outcome
    "events", "description", "bb_type",

    # Score information
    "bat_score", "post_bat_score",
    "home_score", "away_score",

    # Contact variables used for optional extensions
    "launch_speed", "launch_angle",
]

raw_files = sorted(RAW_DIR.glob("statcast_*.csv"))

if not raw_files:
    raise FileNotFoundError(
        "No Statcast files found in data/raw. "
        "Set DOWNLOAD_RAW_DATA = True or place the raw CSV files there."
    )

print(f"Raw Statcast files found: {len(raw_files):,}")

In [ ]:
extra_parts = []

for i, file in enumerate(raw_files, start=1):
    # Read the header first so missing optional columns do not break the loop.
    available = pd.read_csv(file, nrows=0).columns
    usecols = [col for col in EXTRA_COLUMNS if col in available]

    df = pd.read_csv(
        file,
        usecols=usecols,
        low_memory=True
    )

    # The automatic-runner analysis begins in the 10th inning.
    df = df[df["inning"] >= 10].copy()

    if not df.empty:
        extra_parts.append(df)

    if i % 250 == 0:
        print(f"Processed {i:,}/{len(raw_files):,} files")

extra = pd.concat(extra_parts, ignore_index=True)
extra["game_date"] = pd.to_datetime(extra["game_date"])

print(f"Extra-inning pitches: {len(extra):,}")
print(f"Games represented: {extra['game_pk'].nunique():,}")

## 3. Construct one observation per extra-inning half

The main starting state is:

- inning 10 or later,
- runner on second,
- zero outs.

For every qualifying half-inning, we keep the first pitch-state observation and then attach the final outcome of that half-inning and game.

`post_bat_score` is used to calculate runs because it correctly captures the run scored on a walk-off play.

In [ ]:
GROUP_KEYS = ["game_pk", "inning", "inning_topbot"]

# Identify the first pitch of each qualifying half-inning.
extra_start = (
    extra.loc[
        (extra["outs_when_up"] == 0) &
        (extra["on_2b"].notna())
    ]
    .sort_values(GROUP_KEYS + ["at_bat_number", "pitch_number"])
    .groupby(GROUP_KEYS, as_index=False)
    .first()
)

# Score differential from the batting team's perspective.
extra_start["score_diff"] = np.where(
    extra_start["inning_topbot"].eq("Top"),
    extra_start["away_score"] - extra_start["home_score"],
    extra_start["home_score"] - extra_start["away_score"]
)

print(f"Qualifying half-innings: {len(extra_start):,}")

In [ ]:
# Final batting-team score reached during each half-inning.
half_inning_scores = (
    extra
    .groupby(GROUP_KEYS)
    .agg(end_bat_score=("post_bat_score", "max"))
    .reset_index()
)

extra_start = extra_start.merge(
    half_inning_scores,
    on=GROUP_KEYS,
    how="left"
)

# Runs produced from the beginning of the half-inning.
extra_start["runs_scored"] = (
    extra_start["end_bat_score"] - extra_start["bat_score"]
)

for threshold in [1, 2, 3]:
    extra_start[f"scored_{threshold}plus"] = (
        extra_start["runs_scored"] >= threshold
    ).astype(int)

display(
    extra_start[
        GROUP_KEYS + [
            "score_diff", "runs_scored",
            "scored_1plus", "scored_2plus"
        ]
    ].head()
)

In [ ]:
# Build final game scores using post-play batting scores so walk-offs are handled correctly.
game_scores = extra.copy()

game_scores["home_post_score"] = np.where(
    game_scores["inning_topbot"].eq("Bot"),
    game_scores["post_bat_score"],
    game_scores["home_score"]
)

game_scores["away_post_score"] = np.where(
    game_scores["inning_topbot"].eq("Top"),
    game_scores["post_bat_score"],
    game_scores["away_score"]
)

game_results = (
    game_scores
    .groupby("game_pk")
    .agg(
        final_home_score=("home_post_score", "max"),
        final_away_score=("away_post_score", "max")
    )
    .reset_index()
)

extra_start = extra_start.merge(
    game_results,
    on="game_pk",
    how="left"
)

# 1 if the team batting in this half-inning eventually won the game.
extra_start["batting_team_win"] = np.where(
    extra_start["inning_topbot"].eq("Top"),
    extra_start["final_away_score"] > extra_start["final_home_score"],
    extra_start["final_home_score"] > extra_start["final_away_score"]
).astype(int)

## 4. Identify the first-PA strategy

A key correction is to classify **bunt attempts**, not only successful sacrifice bunts.

A batter is marked as attempting a bunt if any pitch in the first plate appearance contains `"bunt"` in the pitch description. This keeps failed bunts, foul bunts, and bunt strikeouts in the bunt group.

In [ ]:
# Flag bunt pitches at the pitch level.
extra["bunt_pitch"] = (
    extra["description"]
    .astype(str)
    .str.contains("bunt", case=False, na=False)
)

# Collapse pitch-level data to one row per plate appearance.
pa_strategy = (
    extra
    .sort_values(GROUP_KEYS + ["at_bat_number", "pitch_number"])
    .groupby(GROUP_KEYS + ["at_bat_number"])
    .agg(
        bunt_pitch=("bunt_pitch", "max"),
        event=("events", "last")
    )
    .reset_index()
)

pa_strategy["bunt_attempt"] = (
    pa_strategy["bunt_pitch"] |
    pa_strategy["event"].eq("sac_bunt")
).astype(int)

# Keep only the first plate appearance of each half-inning.
first_pa_strategy = (
    pa_strategy
    .sort_values(GROUP_KEYS + ["at_bat_number"])
    .groupby(GROUP_KEYS, as_index=False)
    .first()
)

display(
    first_pa_strategy.loc[
        first_pa_strategy["bunt_attempt"].eq(1),
        "event"
    ].value_counts(dropna=False)
)

In [ ]:
# Merge the first-PA strategy onto the half-inning outcome data.
strategy_data = extra_start.merge(
    first_pa_strategy[
        GROUP_KEYS + ["bunt_attempt", "event"]
    ],
    on=GROUP_KEYS,
    how="inner"
)

strategy_data["offensive_strategy"] = np.where(
    strategy_data["bunt_attempt"].eq(1),
    "bunt_attempt",
    "swing_away"
)

print(strategy_data["offensive_strategy"].value_counts())

### Descriptive bunt outcomes

In [ ]:
def classify_bunt_outcome(event):
    if event == "sac_bunt":
        return "sacrifice"
    if event in {"single", "double", "triple"}:
        return "hit"
    if event == "walk":
        return "walk"
    if event == "strikeout":
        return "strikeout"
    if event in {"field_out", "fielders_choice_out", "double_play"}:
        return "other_out"
    return "other"


bunt_pa = first_pa_strategy[
    first_pa_strategy["bunt_attempt"].eq(1)
].copy()

bunt_pa["bunt_outcome"] = bunt_pa["event"].map(classify_bunt_outcome)

bunt_outcomes = (
    bunt_pa["bunt_outcome"]
    .value_counts()
    .rename_axis("outcome")
    .reset_index(name="count")
)

bunt_outcomes["pct"] = (
    100 * bunt_outcomes["count"] / bunt_outcomes["count"].sum()
)

display(bunt_outcomes.round(2))

## 5. Add hitter quality

To reduce selection bias, each extra-inning batter is assigned a **season-to-date wOBA before that game**.

Instead of redownloading another full Statcast dataset, this section streams the existing daily raw files and keeps only completed plate appearances plus the two wOBA fields.

In [ ]:
QUALITY_COLUMNS = [
    "game_date", "game_pk", "batter", "events",
    "at_bat_number", "woba_value", "woba_denom"
]

quality_parts = []

for i, file in enumerate(raw_files, start=1):
    available = pd.read_csv(file, nrows=0).columns
    usecols = [col for col in QUALITY_COLUMNS if col in available]

    # Skip files that cannot support wOBA construction.
    if not {"woba_value", "woba_denom"}.issubset(usecols):
        continue

    df = pd.read_csv(
        file,
        usecols=usecols,
        low_memory=True
    )

    # Statcast records the PA-level event on the final pitch of the PA.
    df = df[df["events"].notna()].copy()

    if not df.empty:
        quality_parts.append(df)

    if i % 250 == 0:
        print(f"Processed hitter-quality file {i:,}/{len(raw_files):,}")

pa_data = pd.concat(quality_parts, ignore_index=True)

pa_data["game_date"] = pd.to_datetime(pa_data["game_date"])
pa_data["woba_value"] = pd.to_numeric(pa_data["woba_value"], errors="coerce")
pa_data["woba_denom"] = pd.to_numeric(pa_data["woba_denom"], errors="coerce")
pa_data["season"] = pa_data["game_date"].dt.year

woba_pa = pa_data[
    pa_data["woba_value"].notna() &
    pa_data["woba_denom"].notna()
].copy()

print(f"Completed PAs used for hitter quality: {len(woba_pa):,}")

In [ ]:
# Aggregate each batter's wOBA contribution by day.
daily_woba = (
    woba_pa
    .groupby(["batter", "season", "game_date"])
    .agg(
        daily_woba_value=("woba_value", "sum"),
        daily_woba_denom=("woba_denom", "sum"),
        daily_pa=("game_pk", "count")
    )
    .reset_index()
    .sort_values(["batter", "season", "game_date"])
)

group = daily_woba.groupby(["batter", "season"])

# Cumulative totals BEFORE the current game date.
daily_woba["prior_woba_value"] = (
    group["daily_woba_value"].cumsum() - daily_woba["daily_woba_value"]
)

daily_woba["prior_woba_denom"] = (
    group["daily_woba_denom"].cumsum() - daily_woba["daily_woba_denom"]
)

daily_woba["prior_pa"] = (
    group["daily_pa"].cumsum() - daily_woba["daily_pa"]
)

daily_woba["season_woba_before_game"] = (
    daily_woba["prior_woba_value"] /
    daily_woba["prior_woba_denom"].replace(0, np.nan)
)

In [ ]:
# Merge pregame hitter quality onto each extra-inning situation.
strategy_data["game_date"] = pd.to_datetime(strategy_data["game_date"])
strategy_data["season"] = strategy_data["game_date"].dt.year

strategy_data = strategy_data.merge(
    daily_woba[
        [
            "batter", "season", "game_date",
            "prior_pa", "season_woba_before_game"
        ]
    ],
    on=["batter", "season", "game_date"],
    how="left"
)

# League-average wOBA from the same sample.
league_woba = (
    woba_pa["woba_value"].sum() /
    woba_pa["woba_denom"].sum()
)

# Shrink noisy early-season values toward league average.
STABILIZATION_PA = 100

strategy_data["adjusted_woba"] = (
    strategy_data["prior_pa"] * strategy_data["season_woba_before_game"]
    + STABILIZATION_PA * league_woba
) / (
    strategy_data["prior_pa"] + STABILIZATION_PA
)

strategy_data["adjusted_woba"] = (
    strategy_data["adjusted_woba"].fillna(league_woba)
)

print(f"League wOBA: {league_woba:.3f}")

## 6. Bunt analysis in tied extra innings

Intentional walks are a defensive decision, so they are excluded from the bunt-versus-swing comparison.

In [ ]:
offensive_data = strategy_data[
    strategy_data["event"].ne("intent_walk")
].copy()

tied_offensive = offensive_data[
    offensive_data["score_diff"].eq(0)
].copy()

bunt_comparison = (
    tied_offensive
    .groupby(["inning_topbot", "offensive_strategy"])
    .agg(
        situations=("game_pk", "count"),
        win_rate=("batting_team_win", "mean"),
        average_runs=("runs_scored", "mean"),
        score_1plus=("scored_1plus", "mean"),
        score_2plus=("scored_2plus", "mean")
    )
    .reset_index()
)

for col in ["win_rate", "score_1plus", "score_2plus"]:
    bunt_comparison[col] *= 100

display(bunt_comparison.round(2))

### Two-proportion tests

In [ ]:
def proportion_test(data, half, outcome):
    subset = data[data["inning_topbot"].eq(half)]

    bunt = subset[subset["offensive_strategy"].eq("bunt_attempt")]
    swing = subset[subset["offensive_strategy"].eq("swing_away")]

    counts = [bunt[outcome].sum(), swing[outcome].sum()]
    nobs = [len(bunt), len(swing)]

    z_stat, p_value = proportions_ztest(counts, nobs)

    return {
        "half": half,
        "outcome": outcome,
        "bunt_rate": bunt[outcome].mean(),
        "swing_rate": swing[outcome].mean(),
        "z_stat": z_stat,
        "p_value": p_value
    }


tests = [
    proportion_test(tied_offensive, half, outcome)
    for half in ["Top", "Bot"]
    for outcome in ["batting_team_win", "scored_1plus"]
]

tests_df = pd.DataFrame(tests)
display(tests_df.round(4))

### Logistic regression with a top/bottom interaction

In [ ]:
tied_offensive["batting_team_win"] = (
    tied_offensive["batting_team_win"].astype(int)
)

bunt_win_model = smf.glm(
    formula=(
        "batting_team_win ~ "
        "bunt_attempt * C(inning_topbot, Treatment(reference='Top')) "
        "+ inning + adjusted_woba"
    ),
    data=tied_offensive,
    family=sm.families.Binomial()
).fit(
    cov_type="cluster",
    cov_kwds={"groups": tied_offensive["game_pk"]}
)

print(bunt_win_model.summary())

In [ ]:
# Convert regression coefficients into interpretable win probabilities
# for a league-average hitter in the 10th inning.
prediction_grid = pd.DataFrame({
    "bunt_attempt": [0, 1, 0, 1],
    "inning_topbot": ["Top", "Top", "Bot", "Bot"],
    "inning": [10, 10, 10, 10],
    "adjusted_woba": [league_woba] * 4
})

prediction_grid["predicted_win_prob"] = (
    bunt_win_model.predict(prediction_grid)
)

prediction_grid["predicted_win_pct"] = (
    100 * prediction_grid["predicted_win_prob"]
)

display(prediction_grid.round(4))

In [ ]:
def probability_effect(predictions, half):
    swing = predictions.loc[
        predictions["inning_topbot"].eq(half) &
        predictions["bunt_attempt"].eq(0),
        "predicted_win_prob"
    ].iloc[0]

    bunt = predictions.loc[
        predictions["inning_topbot"].eq(half) &
        predictions["bunt_attempt"].eq(1),
        "predicted_win_prob"
    ].iloc[0]

    return 100 * (bunt - swing)


top_bunt_effect = probability_effect(prediction_grid, "Top")
bottom_bunt_effect = probability_effect(prediction_grid, "Bot")

print(f"Adjusted top-half bunt effect: {top_bunt_effect:.2f} pp")
print(f"Adjusted bottom-half bunt effect: {bottom_bunt_effect:.2f} pp")

## 7. Empirical strategy engine

The strategy engine uses:

- the **top-half run distribution** under each offensive strategy, and
- the **bottom-half probability of eventually winning** conditional on runs needed.

This avoids treating a walk-off bottom inning as if it were a full three-out inning.

In [ ]:
bottom_data = offensive_data[
    offensive_data["inning_topbot"].eq("Bot")
].copy()

bottom_data["runs_needed"] = 1 - bottom_data["score_diff"]
bottom_data = bottom_data[bottom_data["runs_needed"] >= 1]

bottom_win_by_need = (
    bottom_data
    .groupby(["runs_needed", "offensive_strategy"])
    .agg(
        situations=("game_pk", "count"),
        wins=("batting_team_win", "sum"),
        win_rate=("batting_team_win", "mean")
    )
    .reset_index()
)

bottom_win_by_need["win_pct"] = 100 * bottom_win_by_need["win_rate"]
display(bottom_win_by_need.round(2))

In [ ]:
# Beta(1, 1) smoothing keeps small samples from being treated as exact.
bottom_probs = bottom_win_by_need.copy()

bottom_probs["alpha"] = bottom_probs["wins"] + 1
bottom_probs["beta"] = (
    bottom_probs["situations"] - bottom_probs["wins"] + 1
)

bottom_probs["smoothed_win_prob"] = (
    bottom_probs["alpha"] /
    (bottom_probs["alpha"] + bottom_probs["beta"])
)

bottom_probs["lower_95"] = beta.ppf(
    0.025,
    bottom_probs["alpha"],
    bottom_probs["beta"]
)

bottom_probs["upper_95"] = beta.ppf(
    0.975,
    bottom_probs["alpha"],
    bottom_probs["beta"]
)

display(
    bottom_probs[
        [
            "runs_needed", "offensive_strategy",
            "situations", "wins",
            "smoothed_win_prob", "lower_95", "upper_95"
        ]
    ].round(4)
)

In [ ]:
def build_bottom_lookup(data):
    bottom = data[data["inning_topbot"].eq("Bot")].copy()
    bottom["runs_needed"] = 1 - bottom["score_diff"]
    bottom = bottom[bottom["runs_needed"] >= 1]

    summary = (
        bottom
        .groupby(["runs_needed", "offensive_strategy"])
        .agg(
            situations=("game_pk", "count"),
            wins=("batting_team_win", "sum")
        )
        .reset_index()
    )

    # Beta(1, 1) posterior mean.
    summary["win_prob"] = (
        (summary["wins"] + 1) /
        (summary["situations"] + 2)
    )

    return {
        (int(row["runs_needed"]), row["offensive_strategy"]): row["win_prob"]
        for _, row in summary.iterrows()
    }


def choose_bottom_strategy(runs_needed, policy):
    if policy == "always_swing":
        return "swing_away"

    if policy == "bunt_if_need_1":
        return "bunt_attempt" if runs_needed == 1 else "swing_away"

    if policy == "bunt_if_need_1_or_2":
        return (
            "bunt_attempt"
            if runs_needed in {1, 2}
            else "swing_away"
        )

    raise ValueError(f"Unknown bottom policy: {policy}")


def get_bottom_prob(runs_needed, strategy, lookup):
    # Use the requested strategy when historical evidence exists.
    key = (runs_needed, strategy)
    if key in lookup:
        return lookup[key]

    # Do not extrapolate bunting into deficits with no bunt observations.
    swing_key = (runs_needed, "swing_away")
    if swing_key in lookup:
        return lookup[swing_key]

    # Very large deficits were extremely rare in the observed sample.
    return 0.0

In [ ]:
def calculate_matchup(top_data, bottom_lookup, top_strategy, bottom_policy):
    top_subset = top_data[
        top_data["offensive_strategy"].eq(top_strategy)
    ]

    run_probs = (
        top_subset["runs_scored"]
        .value_counts(normalize=True)
        .sort_index()
    )

    home_win_prob = 0.0

    for away_runs, run_prob in run_probs.items():
        runs_needed = int(away_runs + 1)

        bottom_strategy = choose_bottom_strategy(
            runs_needed,
            bottom_policy
        )

        conditional_home_win = get_bottom_prob(
            runs_needed,
            bottom_strategy,
            bottom_lookup
        )

        home_win_prob += run_prob * conditional_home_win

    return home_win_prob

In [ ]:
top_tied = offensive_data[
    offensive_data["inning_topbot"].eq("Top") &
    offensive_data["score_diff"].eq(0)
].copy()

bottom_lookup = build_bottom_lookup(offensive_data)

TOP_STRATEGIES = ["bunt_attempt", "swing_away"]
BOTTOM_POLICIES = [
    "always_swing",
    "bunt_if_need_1",
    "bunt_if_need_1_or_2"
]

strategy_results = []

for top_strategy in TOP_STRATEGIES:
    for bottom_policy in BOTTOM_POLICIES:
        home_win = calculate_matchup(
            top_tied,
            bottom_lookup,
            top_strategy,
            bottom_policy
        )

        strategy_results.append({
            "top_strategy": top_strategy,
            "bottom_policy": bottom_policy,
            "home_win_pct": 100 * home_win,
            "away_win_pct": 100 * (1 - home_win)
        })

exact_strategy_matrix = pd.DataFrame(strategy_results)
display(exact_strategy_matrix.round(2))

## 8. Game-level bootstrap

This bootstrap resamples entire games, not individual half-innings. That preserves dependence between observations from the same game.

The exact matchup calculation is used inside each bootstrap sample, so there is **no slow inner Monte Carlo loop**.

In [ ]:
def cluster_bootstrap_data(data, rng):
    game_ids = data["game_pk"].unique()

    sampled_games = rng.choice(
        game_ids,
        size=len(game_ids),
        replace=True
    )

    boot_parts = []

    for bootstrap_game_id, game_id in enumerate(sampled_games):
        temp = data[data["game_pk"].eq(game_id)].copy()

        # Repeatedly sampled games receive unique IDs inside the bootstrap sample.
        temp["bootstrap_game_pk"] = bootstrap_game_id
        boot_parts.append(temp)

    return pd.concat(boot_parts, ignore_index=True)


def bootstrap_exact_strategies(data, n_bootstrap=500, seed=42):
    bootstrap_rng = np.random.default_rng(seed)
    records = []

    for b in range(n_bootstrap):
        boot = cluster_bootstrap_data(data, bootstrap_rng)

        boot_top = boot[
            boot["inning_topbot"].eq("Top") &
            boot["score_diff"].eq(0)
        ].copy()

        # Skip the rare resample that loses one of the two top strategies.
        present = set(boot_top["offensive_strategy"].unique())
        if not set(TOP_STRATEGIES).issubset(present):
            continue

        boot_lookup = build_bottom_lookup(boot)

        for top_strategy in TOP_STRATEGIES:
            for bottom_policy in BOTTOM_POLICIES:
                home_win = calculate_matchup(
                    boot_top,
                    boot_lookup,
                    top_strategy,
                    bottom_policy
                )

                records.append({
                    "bootstrap": b,
                    "top_strategy": top_strategy,
                    "bottom_policy": bottom_policy,
                    "home_win_rate": home_win,
                    "away_win_rate": 1 - home_win
                })

    return pd.DataFrame(records)

In [ ]:
bootstrap_results = bootstrap_exact_strategies(
    offensive_data,
    n_bootstrap=500,
    seed=RANDOM_SEED
)

bootstrap_summary = (
    bootstrap_results
    .groupby(["top_strategy", "bottom_policy"])
    .agg(
        mean_home_win=("home_win_rate", "mean"),
        lower_95=("home_win_rate", lambda x: x.quantile(0.025)),
        upper_95=("home_win_rate", lambda x: x.quantile(0.975))
    )
    .reset_index()
)

for col in ["mean_home_win", "lower_95", "upper_95"]:
    bootstrap_summary[col] *= 100

display(bootstrap_summary.round(2))

In [ ]:
# Visitor effect: swing away vs bunt when the home team bunts only if one run is needed.
visitor_boot = (
    bootstrap_results[
        bootstrap_results["bottom_policy"].eq("bunt_if_need_1")
    ]
    .pivot(
        index="bootstrap",
        columns="top_strategy",
        values="away_win_rate"
    )
    .dropna()
)

visitor_effect = 100 * (
    visitor_boot["swing_away"] - visitor_boot["bunt_attempt"]
)

# Home effect: bunt only if one run is needed vs always swing,
# holding the visitor strategy at swing away.
home_boot = (
    bootstrap_results[
        bootstrap_results["top_strategy"].eq("swing_away")
    ]
    .pivot(
        index="bootstrap",
        columns="bottom_policy",
        values="home_win_rate"
    )
    .dropna()
)

home_effect = 100 * (
    home_boot["bunt_if_need_1"] - home_boot["always_swing"]
)

print(
    f"Visitor swing-away advantage: {visitor_effect.mean():.2f} pp "
    f"(95% CI: {visitor_effect.quantile(.025):.2f} to "
    f"{visitor_effect.quantile(.975):.2f})"
)

print(
    f"Home bunt-if-need-1 advantage: {home_effect.mean():.2f} pp "
    f"(95% CI: {home_effect.quantile(.025):.2f} to "
    f"{home_effect.quantile(.975):.2f})"
)

## 9. Exploratory hitter-quality groups

These subgroup results are descriptive because the bunt samples become small after splitting the data.

In [ ]:
# Divide adjusted wOBA into three equally sized groups.
tied_offensive["hitter_group"] = pd.qcut(
    tied_offensive["adjusted_woba"],
    q=3,
    labels=["Low wOBA", "Average wOBA", "High wOBA"],
    duplicates="drop"
)

hitter_strategy_summary = (
    tied_offensive
    .groupby(
        ["inning_topbot", "hitter_group", "offensive_strategy"],
        observed=True
    )
    .agg(
        situations=("game_pk", "count"),
        win_rate=("batting_team_win", "mean"),
        avg_runs=("runs_scored", "mean"),
        score_1plus=("scored_1plus", "mean"),
        score_2plus=("scored_2plus", "mean")
    )
    .reset_index()
)

for col in ["win_rate", "score_1plus", "score_2plus"]:
    hitter_strategy_summary[col] *= 100

display(hitter_strategy_summary.round(2))

In [ ]:
def fisher_strategy_test(data, half, hitter_group, outcome):
    subset = data[
        data["inning_topbot"].eq(half) &
        data["hitter_group"].eq(hitter_group)
    ]

    bunt = subset[subset["offensive_strategy"].eq("bunt_attempt")]
    swing = subset[subset["offensive_strategy"].eq("swing_away")]

    table = [
        [bunt[outcome].sum(), len(bunt) - bunt[outcome].sum()],
        [swing[outcome].sum(), len(swing) - swing[outcome].sum()]
    ]

    _, p_value = fisher_exact(table)

    return {
        "half": half,
        "hitter_group": hitter_group,
        "outcome": outcome,
        "bunt_n": len(bunt),
        "swing_n": len(swing),
        "bunt_rate": bunt[outcome].mean(),
        "swing_rate": swing[outcome].mean(),
        "difference_pp": 100 * (
            bunt[outcome].mean() - swing[outcome].mean()
        ),
        "p_value": p_value
    }


group_tests = pd.DataFrame([
    fisher_strategy_test(
        tied_offensive,
        half,
        hitter_group,
        "batting_team_win"
    )
    for half in ["Top", "Bot"]
    for hitter_group in ["Low wOBA", "Average wOBA"]
])

display(group_tests.round(3))

In [ ]:
scoring_tests = pd.DataFrame([
    fisher_strategy_test(
        tied_offensive,
        "Bot",
        hitter_group,
        "scored_1plus"
    )
    for hitter_group in ["Low wOBA", "Average wOBA"]
])

display(scoring_tests.round(3))

## 10. Intentional-walk analysis

The intentional walk is treated as a **defensive** strategy. The primary sample is tied bottom-half situations, where one run wins the game.

In [ ]:
iwalk_data = strategy_data.copy()

iwalk_data["intentional_walk"] = (
    iwalk_data["event"].eq("intent_walk")
).astype(int)

iwalk_data["defensive_strategy"] = np.where(
    iwalk_data["intentional_walk"].eq(1),
    "intent_walk",
    "pitch_to_hitter"
)

bottom_iwalk = iwalk_data[
    iwalk_data["score_diff"].eq(0) &
    iwalk_data["inning_topbot"].eq("Bot")
].copy()

bottom_iwalk["batting_team_win"] = (
    bottom_iwalk["batting_team_win"].astype(int)
)

bottom_iwalk["scored_1plus"] = (
    bottom_iwalk["scored_1plus"].astype(int)
)

bottom_iwalk["adjusted_woba"] = (
    bottom_iwalk["adjusted_woba"].fillna(league_woba)
)

iwalk_summary = (
    bottom_iwalk
    .groupby("defensive_strategy")
    .agg(
        situations=("game_pk", "count"),
        batting_team_win=("batting_team_win", "mean"),
        average_runs=("runs_scored", "mean"),
        score_1plus=("scored_1plus", "mean")
    )
    .reset_index()
)

for col in ["batting_team_win", "score_1plus"]:
    iwalk_summary[col] *= 100

display(iwalk_summary.round(2))

In [ ]:
iwalk = bottom_iwalk[
    bottom_iwalk["defensive_strategy"].eq("intent_walk")
]

pitch = bottom_iwalk[
    bottom_iwalk["defensive_strategy"].eq("pitch_to_hitter")
]

win_table = [
    [
        iwalk["batting_team_win"].sum(),
        len(iwalk) - iwalk["batting_team_win"].sum()
    ],
    [
        pitch["batting_team_win"].sum(),
        len(pitch) - pitch["batting_team_win"].sum()
    ]
]

score_table = [
    [
        iwalk["scored_1plus"].sum(),
        len(iwalk) - iwalk["scored_1plus"].sum()
    ],
    [
        pitch["scored_1plus"].sum(),
        len(pitch) - pitch["scored_1plus"].sum()
    ]
]

_, p_win = fisher_exact(win_table)
_, p_score = fisher_exact(score_table)

print(f"Eventual win-rate Fisher p-value: {p_win:.4f}")
print(f"Score-1+ Fisher p-value: {p_score:.4f}")

In [ ]:
iwalk_score_model = smf.glm(
    formula="scored_1plus ~ intentional_walk + adjusted_woba + inning",
    data=bottom_iwalk,
    family=sm.families.Binomial()
).fit(
    cov_type="cluster",
    cov_kwds={"groups": bottom_iwalk["game_pk"]}
)

print(iwalk_score_model.summary())

In [ ]:
iwalk_prediction = pd.DataFrame({
    "intentional_walk": [0, 1],
    "adjusted_woba": [league_woba, league_woba],
    "inning": [10, 10]
})

iwalk_prediction["score_probability"] = (
    iwalk_score_model.predict(iwalk_prediction)
)

iwalk_prediction["score_pct"] = (
    100 * iwalk_prediction["score_probability"]
)

iwalk_prediction["strategy"] = [
    "Pitch to hitter",
    "Intentional walk"
]

display(iwalk_prediction[["strategy", "score_pct"]].round(2))

## 11. Final figures and outputs

In [ ]:
# Strategy recommendation table used in the final write-up.
strategy_chart = pd.DataFrame({
    "Situation": [
        "Top half, tied",
        "Bottom half, need 1 run",
        "Bottom half, need 2+ runs",
        "Bottom half, intentional-walk decision"
    ],
    "Recommended strategy": [
        "Swing away",
        "Bunt selectively",
        "Swing away",
        "Pitch to hitter"
    ],
    "Evidence": [
        "Point estimate favors swing-away; bootstrap CI crosses 0",
        "Home bunt policy improves win probability by about 4.5 pp",
        "Preserve outs and multi-run upside",
        "No measured defensive scoring benefit from the IBB"
    ]
})

display(strategy_chart)

In [ ]:
# Figure 1: adjusted bunt effect by inning half.
bunt_effects = pd.DataFrame({
    "Situation": ["Top half", "Bottom half"],
    "Effect": [top_bunt_effect, bottom_bunt_effect]
})

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(bunt_effects["Situation"], bunt_effects["Effect"])
ax.axhline(0, linewidth=1)
ax.set_ylabel("Change in win probability (percentage points)")
ax.set_title("Adjusted Effect of a Bunt Attempt in Tied Extra Innings")

for i, value in enumerate(bunt_effects["Effect"]):
    ax.text(
        i,
        value,
        f"{value:+.2f} pp",
        ha="center",
        va="bottom" if value >= 0 else "top"
    )

fig.tight_layout()
fig.savefig(
    FIGURE_DIR / "bunt_win_probability_effect.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

In [ ]:
# Figure 2: bootstrap interval for the home bunt-if-need-1 policy.
home_mean = home_effect.mean()
home_lower = home_effect.quantile(0.025)
home_upper = home_effect.quantile(0.975)

fig, ax = plt.subplots(figsize=(8, 4))
ax.errorbar(
    x=home_mean,
    y=0,
    xerr=[[home_mean - home_lower], [home_upper - home_mean]],
    fmt="o",
    capsize=6
)
ax.axvline(0, linewidth=1)
ax.set_yticks([0])
ax.set_yticklabels(["Bunt when exactly 1 run is needed"])
ax.set_xlabel("Change in home win probability (percentage points)")
ax.set_title("Bootstrap Estimate of the Home-Team Bunt Advantage")

fig.tight_layout()
fig.savefig(
    FIGURE_DIR / "home_bunt_bootstrap_ci.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

In [ ]:
# Figure 3: predicted scoring probability under the intentional-walk decision.
fig, ax = plt.subplots(figsize=(7, 5))
ax.bar(
    iwalk_prediction["strategy"],
    iwalk_prediction["score_pct"]
)
ax.set_ylabel("Estimated probability offense scores (%)")
ax.set_title("Intentional Walk Strategy in the Bottom of the 10th")
ax.set_ylim(0, 80)

for i, value in enumerate(iwalk_prediction["score_pct"]):
    ax.text(i, value + 1, f"{value:.1f}%", ha="center")

fig.tight_layout()
fig.savefig(
    FIGURE_DIR / "intentional_walk_scoring_probability.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

In [ ]:
# Save the key datasets and result tables for the write-up.
strategy_data.to_csv(
    PROCESSED_DIR / "extra_innings_strategy.csv",
    index=False
)

bunt_comparison.to_csv(
    RESULTS_DIR / "bunt_comparison.csv",
    index=False
)

exact_strategy_matrix.to_csv(
    RESULTS_DIR / "strategy_matrix.csv",
    index=False
)

bootstrap_summary.to_csv(
    RESULTS_DIR / "bootstrap_strategy_summary.csv",
    index=False
)

iwalk_summary.to_csv(
    RESULTS_DIR / "intentional_walk_summary.csv",
    index=False
)

strategy_chart.to_csv(
    RESULTS_DIR / "strategy_recommendations.csv",
    index=False
)

print("Analysis outputs saved.")

## Main takeaway

The analysis does **not** support a universal rule that bunting is always good or always bad in extra innings.

The stronger pattern is state dependent:

- **Top half:** preserve outs; swing-away has the better point estimate, though uncertainty remains.
- **Bottom half, exactly one run needed:** a selective bunt policy has the strongest evidence of improving win probability.
- **Bottom half, multiple runs needed:** preserve outs and multi-run upside.
- **Intentional walk:** no clear evidence of a defensive scoring benefit.

The key strategic variable is the **value of the next run**.